In [8]:
import pandas as pd
import sqlite3

In [9]:
df_2009 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2010 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
df_comb = pd.concat([df_2009,df_2010], ignore_index=True)

In [10]:
df_comb.info()


<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [11]:
#we could assume that the missing value of "Customer ID" stands for non regitred 
# clients let's name it non_registred
#we have some missing Description we could try to copy the description of the same 
#StockCode if needed if it is new we could just keep it 
x = df_comb.isna().sum() / len(df_comb) * 100
print(x)

Invoice         0.000000
StockCode       0.000000
Description     0.410541
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    22.766873
Country         0.000000
dtype: float64


In [12]:
int((df_comb["Quantity"] < 0).sum())

22950

In [13]:
df_comb["Customer ID"] = df_comb["Customer ID"].fillna(0).astype(int)
df_comb["revenue"] = df_comb["Quantity"] * df_comb["Price"]
df_comb.columns = df_comb.columns.str.lower().str.replace(" ", "_")

In [14]:
conn = sqlite3.connect("online_retail.db")
 

In [ ]:
transactions = df_comb[
    [
        "invoice",
        "stockcode",
        "customer_id",
        "invoicedate",
        "quantity",
        "price",
        "revenue"
    ]
].copy()


# customers table: only real, identifiable customers
customers = (
    df_comb[df_comb["customer_id"] != 0]
    .groupby("customer_id")
    .agg(
        country=("country", "first"),   # still worth double-checking this is stable per real customer
        first_seen=("invoicedate", "min"),
        last_seen=("invoicedate", "max")
    )
    .reset_index()
)

guest_by_country = (
    df_comb[df_comb["customer_id"] == 0]
    .groupby("country")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

products = (
    df_comb[df_comb["price"] > 0]          # drop zero/negative-price noise
    .groupby("stockcode")
    .agg(
        description=("description", lambda x: x.dropna().mode().iloc[0] if x.notna().any() else None),
        avg_price=("price", "median"),      # more robust than mean to one-off outlier prices
    )
    .reset_index()
)

In [15]:
#What percentage of gross revenue is lost due to returns and cancellations?
#returns quantity < 0

returns_query = """
SELECT SUM(revenue) AS returned_revenue
FROM retail_data
WHERE quantity < 0
"""

returns = pd.read_sql_query(returns_query, conn)
returns

,returned_revenue
0,-1527041.43


In [16]:
gross_query = """
SELECT SUM(revenue) AS gross_revenue
FROM retail_data
WHERE quantity > 0
"""

gross = pd.read_sql_query(gross_query, conn)
gross

,gross_revenue
0,2.081429e+07


In [17]:
revenue_lost = abs(returns["returned_revenue"].iloc[0])
gross_revenue = gross["gross_revenue"].iloc[0]

leakage_percentage = (revenue_lost / gross_revenue) * 100

print(f"Gross Revenue: {gross_revenue:,.2f}")
print(f"Revenue Lost: {revenue_lost:,.2f}")
print(f"Revenue Leakage: {leakage_percentage:.2f}%")

Gross Revenue: 20,814,292.00
Revenue Lost: 1,527,041.43
Revenue Leakage: 7.34%


##Which products and countries contribute the most to revenue leakage?


In [18]:
revenue_countries_query = """
SELECT
    country,
    ABS(SUM(revenue)) AS revenue_lost
FROM retail_data
WHERE quantity < 0
GROUP BY country
ORDER BY revenue_lost DESC
"""

revenue_countries = pd.read_sql_query(revenue_countries_query, conn)

revenue_countries

,country,revenue_lost
0,United Kingdom,1330091.31
1,EIRE,48912.23
2,France,28752.80
3,Norway,20866.59
4,Spain,17319.05
5,Germany,13273.90
6,Singapore,12158.90
7,Hong Kong,9855.02
8,Netherlands,5707.39
9,Portugal,4879.85


In [19]:
revenue_products_query = """
SELECT
    StockCode,
    description,
    ABS(SUM(revenue)) AS revenue_lost
FROM retail_data
WHERE quantity < 0
GROUP BY StockCode
ORDER BY revenue_lost DESC
"""
revenue_products = pd.read_sql_query(revenue_products_query, conn)
revenue_products.head(10)   

,stockcode,description,revenue_lost
0,M,Manual,423886.17
1,AMAZONFEE,AMAZON FEE,294772.71
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
3,23166,MEDIUM CERAMIC TOP STORAGE JAR,77479.64
4,BANK CHARGES,Bank Charges,36096.87
5,22423,REGENCY CAKESTAND 3 TIER,16749.60
6,POST,POSTAGE,15256.42
7,D,Discount,13882.43
8,85123A,21733 mixed,9389.65
9,CRUK,CRUK Commission,7933.43
